In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import torchvision.utils as vutils
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
latent_dim = 128
batch_size = 128
epochs = 25

In [50]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(root='../stargan-v2/data/afhq/train', transform=transform)
val_dataset = datasets.ImageFolder(root='../stargan-v2/data/afhq/val', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, drop_last=False)

In [51]:
class ConvVAE(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1), nn.ReLU(),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(256 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(256 * 4 * 4, latent_dim)
        
        self.fc_dec = nn.Linear(latent_dim, 256 * 4 * 4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid()
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.fc_dec(z).view(-1, 256, 4, 4)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

In [ ]:
def loss_function(recon_x, x, mu, logvar, beta):
    BCE = F.mse_loss(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + beta * KLD

def train_model(beta_val):
    model = ConvVAE(latent_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    writer = SummaryWriter(f"runs/ConvVAE_beta_{beta_val}")
    
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for imgs, _ in tqdm(train_loader, desc=f"Train Ep {epoch} (Beta={beta_val})", leave=False):
            imgs = imgs.to(device)
            optimizer.zero_grad()
            recon_batch, mu, logvar = model(imgs)
            loss = loss_function(recon_batch, imgs, mu, logvar, beta_val)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        avg_train_loss = train_loss / len(train_dataset)
        history['train_loss'].append(avg_train_loss)
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for imgs_val, _ in tqdm(val_loader, desc=f"Val Ep {epoch} (Beta={beta_val})", leave=False):
                imgs_val = imgs_val.to(device)
                recon_val, mu_val, logvar_val = model(imgs_val)
                loss_v = loss_function(recon_val, imgs_val, mu_val, logvar_val, beta_val)
                val_loss += loss_v.item()
                
        avg_val_loss = val_loss / len(val_dataset)
        history['val_loss'].append(avg_val_loss)
        
        writer.add_scalar('Loss/Train', avg_train_loss, epoch)
        writer.add_scalar('Loss/Validation', avg_val_loss, epoch)
        writer.add_images('Reconstructions', torch.cat([imgs_val[:8], recon_val[:8]], 0), epoch)
        
        print(f"Beta: {beta_val} | Ep: {epoch} | Train Loss: {avg_train_loss:.2f} | Val Loss: {avg_val_loss:.2f}")
    
    writer.close()
    
    return {
        'model': model,
        'history': history,
        'beta': beta_val
    }

betas = [0.05, 0.1, 1.0, 10.0]
results = {b: train_model(b) for b in betas}

Train Ep 0 (Beta=0.1):   0%|          | 0/114 [00:00<?, ?it/s]

Beta: 0.1 | Ep: 0 | Train Loss: 490.08 | Val Loss: 322.43


Beta: 0.1 | Ep: 1 | Train Loss: 287.33 | Val Loss: 256.56


Beta: 0.1 | Ep: 2 | Train Loss: 240.59 | Val Loss: 215.60


Beta: 0.1 | Ep: 3 | Train Loss: 206.09 | Val Loss: 194.87


...

Beta: 5.0 | Ep: 2 | Train Loss: 414.01 | Val Loss: 396.10


Beta: 5.0 | Ep: 3 | Train Loss: 390.15 | Val Loss: 380.91


Beta: 5.0 | Ep: 4 | Train Loss: 379.21 | Val Loss: 371.76


Beta: 5.0 | Ep: 5 | Train Loss: 370.66 | Val Loss: 365.90


KeyboardInterrupt: 

In [ ]:
def count_model_params(model):
    """ Counting the number of learnable parameters in a nn.Module """
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return num_params


def smooth(f, K=5):
    """ Smoothing a function using a low-pass filter (mean) of size K """
    kernel = np.ones(K) / K
    f = np.concatenate([f[:int(K//2)], f, f[int(-K//2):]])
    smooth_f = np.convolve(f, kernel, mode="same")
    smooth_f = smooth_f[K//2: -K//2]
    return smooth_f

In [ ]:
def plot_training_progress(loss_iters, title="Training Progress"):
    """ Plotting per-iteration training loss on linear and log scale """
    plt.style.use('seaborn-v0_8')
    fig, ax = plt.subplots(1, 2)
    fig.set_size_inches(18, 5)

    smooth_loss = smooth(loss_iters, 31)
    for a, scale in zip(ax, ["linear", "log"]):
        a.plot(loss_iters,  c="blue", label="Loss",         linewidth=3, alpha=0.5)
        a.plot(smooth_loss, c="red",  label="Smoothed Loss", linewidth=3)
        a.legend(loc="best")
        a.set_xlabel("Iteration")
        a.set_ylabel("CE Loss")
        a.set_title(f"{title} ({scale} scale)")
        if scale == "log":
            a.set_yscale("log")
    plt.show()


def plot_loss_curves(stats, title=""):
    """ Plotting train/val loss and validation accuracy curves per epoch """
    plt.style.use('seaborn-v0_8')
    fig, ax = plt.subplots(1, 2)
    fig.set_size_inches(18, 5)

    epochs = np.arange(1, len(stats["train_loss"]) + 1)
    ax[0].plot(epochs, stats["train_loss"], c="red",  label="Train Loss", linewidth=3)
    ax[0].plot(epochs, stats["val_loss"],   c="blue", label="Valid Loss", linewidth=3)
    ax[0].legend(loc="best")
    ax[0].set_xlabel("Epoch")
    ax[0].set_ylabel("CE Loss")
    ax[0].set_title(f"Loss Curves  {title}")

    best_acc = max(stats["valid_acc"])
    best_ep  = stats["valid_acc"].index(best_acc) + 1
    ax[1].plot(epochs, stats["valid_acc"], c="red", linewidth=3)
    ax[1].set_xlabel("Epoch")
    ax[1].set_ylabel("Accuracy (%)")
    ax[1].set_title(f"Validation Accuracy  (max={round(best_acc, 2)}% @ epoch {best_ep})  {title}")
    plt.show()

In [ ]:
for b in betas:
    plot_training_progress(results[b]["history"], title=f"Beta={b}")
    plot_loss_curves(results[b]["history"], title=f"Beta={b}")

In [ ]:
def generate_random_images(model, num_images=16):
    model.eval()
    with torch.no_grad():
        z = torch.randn(num_images, latent_dim).to(device)
        gen_imgs = model.decode(z)
        
        grid = vutils.make_grid(gen_imgs, nrow=4, padding=2)
        plt.figure(figsize=(6, 6))
        plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
        plt.title("Randomly Sampled Images")
        plt.axis('off')
        plt.show()

def visualize_interpolation(model, n_steps=10):
    model.eval()
    with torch.no_grad():
        z1 = torch.randn(1, latent_dim).to(device)
        z2 = torch.randn(1, latent_dim).to(device)
        alphas = torch.linspace(0, 1, n_steps).view(-1, 1).to(device)
        
        z_interp = z1 * (1 - alphas) + z2 * alphas
        imgs = model.decode(z_interp)
        
        grid = vutils.make_grid(imgs, nrow=n_steps, padding=2)
        plt.figure(figsize=(15, 3))
        plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
        plt.title("Latent Space Interpolation")
        plt.axis('off')
        plt.show()


for b in betas:
    best_model = results[b]['model']

    print(f"1. Generating new images from random latent vectors (Beta={b})...")
    generate_random_images(best_model, num_images=16)
    
    print(f"2. Investigating latent space via interpolation (Beta={b})...")
    visualize_interpolation(best_model, n_steps=10)

In [ ]:
class ConditionalConvVAE(nn.Module):
    def __init__(self, latent_dim=128, num_classes=3):
        super().__init__()
        self.num_classes = num_classes
        self.embed = nn.Embedding(num_classes, 64 * 64)
        
        self.encoder = nn.Sequential(
            nn.Conv2d(4, 32, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1), nn.ReLU(),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(256 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(256 * 4 * 4, latent_dim)
        
        self.fc_dec = nn.Linear(latent_dim + num_classes, 256 * 4 * 4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid()
        )

    def encode(self, x, c):
        c_emb = self.embed(c).view(-1, 1, 64, 64)
        x = torch.cat([x, c_emb], dim=1)
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def decode(self, z, c):
        c_onehot = F.one_hot(c, self.num_classes).float()
        z = torch.cat([z, c_onehot], dim=1)
        h = self.fc_dec(z).view(-1, 256, 4, 4)
        return self.decoder(h)

    def forward(self, x, c):
        mu, logvar = self.encode(x, c)
        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std
        return self.decode(z, c), mu, logvar

In [ ]:
cvae = ConditionalConvVAE(latent_dim).to(device)
optimizer_c = optim.Adam(cvae.parameters(), lr=1e-3)

for epoch in range(epochs):
    cvae.train()
    train_loss = 0
    for imgs, labels in tqdm(train_loader, desc=f"CVAE Train Ep {epoch}", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer_c.zero_grad()
        recon, mu, logvar = cvae(imgs, labels)
        loss = loss_function(recon, imgs, mu, logvar, beta=1.0)
        loss.backward()
        optimizer_c.step()
        train_loss += loss.item()
        
    avg_train_loss = train_loss / len(train_dataset)
    
    cvae.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs_val, labels_val in tqdm(val_loader, desc=f"CVAE Val Ep {epoch}", leave=False):
            imgs_val, labels_val = imgs_val.to(device), labels_val.to(device)
            recon_val, mu_val, logvar_val = cvae(imgs_val, labels_val)
            loss_v = loss_function(recon_val, imgs_val, mu_val, logvar_val, beta=1.0)
            val_loss += loss_v.item()
            
    avg_val_loss = val_loss / len(val_dataset)
    
    print(f"CVAE Epoch {epoch} | Train Loss: {avg_train_loss:.2f} | Val Loss: {avg_val_loss:.2f}")

cvae.eval()
with torch.no_grad():
    z = torch.randn(3, latent_dim).to(device)
    c = torch.tensor([0, 1, 2]).to(device) 
    gen_imgs = cvae.decode(z, c)
    
    grid = vutils.make_grid(gen_imgs, nrow=3)
    plt.figure(figsize=(6, 3))
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
    plt.title("Generazione Condizionata: Cat (0) | Dog (1) | Wildlife (2)")
    plt.axis('off')
    plt.show()